# Fraud detection model vs. rule-based baseline

* **Data**: `ml/data/features_transactions.parquet` (one-off export of the dbt model `features_transactions`, with the `dataset_split` of `fraud_rules_baseline`).
* **Protocol**: train on `train` (days 1-40), evaluate on `test` (days 41-50). `demo` (days 51-60) is **not used**.
* **Decision threshold** chosen on a time-based validation slice of `train` (last 20% of the train period), never on `test`.
* **Excluded from the features**: the label `is_fraud`, `fraud_type` (evaluation only), identifiers, the timestamp, the split and the rule columns.
* **Baseline to beat**: rules F1 = 0.77 on `test`.

In [1]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
ML_DIR = Path(os.getenv("ML_DIR", "/home/jovyan/ml"))
df = pd.read_parquet(ML_DIR / "data" / "features_transactions.parquet")
df = df.sort_values(["transaction_time", "transaction_id"]).reset_index(drop=True)
df.groupby("dataset_split")["is_fraud"].agg(rows="size", frauds="sum", fraud_rate="mean")

,rows,frauds,fraud_rate
dataset_split,,,
demo,17096,223,0.013044
test,17012,163,0.009581
train,67978,654,0.009621


## 1. Features and time-based split
Numeric behaviour features + one-hot encoded categories. Everything else is excluded, and an assertion checks that no forbidden column slipped in.

In [2]:
NUMERIC = [
    "amount", "avg_amount_past", "amount_vs_avg", "nb_tx_past", "hour_of_day", "is_night",
    "is_new_device", "device_age_hours", "is_new_country", "nb_tx_10min", "nb_tx_1h",
    "is_new_recipient", "nb_senders_to_recipient_24h",
]
CATEGORICAL = ["transaction_type", "channel", "merchant_category"]
EXCLUDED = ["is_fraud", "fraud_type", "transaction_id", "customer_id", "account_id", "transaction_time",
            "dataset_split", "rule_burst", "rule_micro_amount", "rule_new_device_night", "rule_alert"]
assert not set(NUMERIC + CATEGORICAL) & set(EXCLUDED), "a forbidden column is used as a feature"
assert set(NUMERIC + CATEGORICAL + EXCLUDED) == set(df.columns), set(df.columns) ^ set(NUMERIC + CATEGORICAL + EXCLUDED)

train = df[df.dataset_split == "train"]
test = df[df.dataset_split == "test"]           # demo is never touched

# validation slice = last 20% of the TRAIN period (time-based, like the real future)
cut = train.transaction_time.quantile(0.8)
fit, val = train[train.transaction_time < cut], train[train.transaction_time >= cut]

CATEGORIES = {c: sorted(fit[c].dropna().unique()) for c in CATEGORICAL}   # learned on fit only


def make_X(frame):
    X = frame[NUMERIC].astype(float).copy()
    for c in CATEGORICAL:
        cat = pd.Categorical(frame[c], categories=CATEGORIES[c])
        X = X.join(pd.get_dummies(cat, prefix=c, dtype=float).set_index(frame.index))
    return X


X_fit, X_val, X_test = make_X(fit), make_X(val), make_X(test)
y_fit, y_val, y_test = fit.is_fraud.astype(int), val.is_fraud.astype(int), test.is_fraud.astype(int)
FEATURES = list(X_fit.columns)
pd.DataFrame({
    "period": [f"{fit.transaction_time.min():%Y-%m-%d} -> {cut:%Y-%m-%d %H:%M}",
               f"{cut:%Y-%m-%d %H:%M} -> {val.transaction_time.max():%Y-%m-%d}",
               f"{test.transaction_time.min():%Y-%m-%d} -> {test.transaction_time.max():%Y-%m-%d}"],
    "rows": [len(fit), len(val), len(test)], "frauds": [y_fit.sum(), y_val.sum(), y_test.sum()],
}, index=["fit (train 80%)", "validation (train 20%)", "test"]).assign(n_features=len(FEATURES))

,period,rows,frauds,n_features
fit (train 80%),2026-09-12 -> 2026-10-14 16:18,54382,568,32
validation (train 20%),2026-10-14 16:18 -> 2026-10-22,13596,86,32
test,2026-10-22 -> 2026-11-01,17012,163,32


## 2. Helpers: threshold choice and metrics
The threshold maximises F1 **on the validation slice**. PR-AUC (average precision) summarises the model over all thresholds and is the right metric with ~1% fraud.

In [3]:
def best_threshold(y, proba):
    p, r, t = precision_recall_curve(y, proba)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[np.argmax(f1)])


def scores(y, pred, proba=None):
    return {
        "alerts": int(pred.sum()), "true alerts": int((pred & (y == 1)).sum()),
        "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred),
        "F1": f1_score(y, pred), "PR-AUC": average_precision_score(y, proba) if proba is not None else np.nan,
    }


def recall_by_scenario(frame, pred):
    frauds = frame.assign(caught=np.asarray(pred, dtype=bool))[frame.is_fraud]
    return frauds.groupby("fraud_type")["caught"].agg(lambda s: f"{s.sum()}/{len(s)} = {s.mean():.0%}")

## 3. Logistic regression
Linear baseline: missing history is imputed with train medians, skewed counts/amounts are log-transformed, features are standardised. `class_weight="balanced"` handles the 1% fraud rate.

In [4]:
LOG_COLS = ["amount", "avg_amount_past", "amount_vs_avg", "nb_tx_past", "device_age_hours",
            "nb_tx_10min", "nb_tx_1h", "nb_senders_to_recipient_24h"]
MEDIANS = X_fit[NUMERIC].median()      # learned on fit only


def prep_lr(X):
    X = X.copy()
    X["has_history"] = X["avg_amount_past"].notna().astype(float)
    X[NUMERIC] = X[NUMERIC].fillna(MEDIANS)
    X[LOG_COLS] = np.log1p(X[LOG_COLS].clip(lower=0))
    return X


logreg = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000, random_state=SEED))
logreg.fit(prep_lr(X_fit), y_fit)
thr_lr = best_threshold(y_val, logreg.predict_proba(prep_lr(X_val))[:, 1])
proba_lr = logreg.predict_proba(prep_lr(X_test))[:, 1]
pred_lr = proba_lr >= thr_lr
print(f"threshold chosen on validation: {thr_lr:.3f}")

threshold chosen on validation: 0.991


## 4. XGBoost
Gradient-boosted trees handle missing values natively and learn interactions (e.g. *new device* **and** *night*). `scale_pos_weight` = #legit / #fraud compensates the imbalance; early stopping on the validation slice (PR-AUC) avoids overfitting.

In [5]:
spw = float((y_fit == 0).sum() / (y_fit == 1).sum())
XGB_PARAMS = {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 4, "subsample": 0.8, "colsample_bytree": 0.8,
                  "min_child_weight": 1, "scale_pos_weight": spw, "eval_metric": "aucpr", "early_stopping_rounds": 50,
                  "random_state": SEED, "n_jobs": -1}
model_xgb = xgb.XGBClassifier(**XGB_PARAMS)
model_xgb.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
thr_xgb = best_threshold(y_val, model_xgb.predict_proba(X_val)[:, 1])
proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
pred_xgb = proba_xgb >= thr_xgb
print(f"scale_pos_weight={spw:.0f} | trees kept={model_xgb.best_iteration + 1} | threshold chosen on validation: {thr_xgb:.3f}")

scale_pos_weight=95 | trees kept=95 | threshold chosen on validation: 0.954


## 5. Results on the test period: rules vs. logistic regression vs. XGBoost

In [6]:
pred_rules = test.rule_alert.to_numpy() == 1
results = pd.DataFrame({
    "Rules (baseline)": scores(y_test, pred_rules),
    "Logistic regression": scores(y_test, pred_lr, proba_lr),
    "XGBoost": scores(y_test, pred_xgb, proba_xgb),
}).T
results.round(3)

,alerts,true alerts,precision,recall,F1,PR-AUC
Rules (baseline),154.0,122.0,0.792,0.748,0.770,NaN
Logistic regression,157.0,139.0,0.885,0.853,0.869,0.938
XGBoost,150.0,134.0,0.893,0.822,0.856,0.942


In [7]:
pd.DataFrame({
    "Rules": recall_by_scenario(test, pred_rules),
    "Logistic regression": recall_by_scenario(test, pred_lr),
    "XGBoost": recall_by_scenario(test, pred_xgb),
}).rename_axis("recall by scenario (test)")

,Rules,Logistic regression,XGBoost
recall by scenario (test),,,
account_takeover,2/21 = 10%,19/21 = 90%,9/21 = 43%
balance_draining,7/19 = 37%,13/19 = 68%,8/19 = 42%
card_testing,92/94 = 98%,80/94 = 85%,91/94 = 97%
rapid_transfers,21/29 = 72%,27/29 = 93%,26/29 = 90%


## 6. What XGBoost relies on
Importance by **gain** = how much each feature improves the trees' splits on average.

In [8]:
importance = pd.Series(model_xgb.get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0)
(100 * importance / importance.sum()).sort_values(ascending=False).head(12).round(1).rename("share of total gain (%)").to_frame()

,share of total gain (%)
nb_tx_1h,38.9
nb_tx_10min,12.2
device_age_hours,4.7
nb_senders_to_recipient_24h,4.5
transaction_type_TRANSFER,4.3
is_night,2.9
amount,2.7
amount_vs_avg,2.6
hour_of_day,2.5
nb_tx_past,2.5


## 7. Generalisation test: can the model catch a scenario it never saw?
XGBoost is retrained with the same settings **without any `card_testing` fraud** in fit/validation (those rows are removed; `fraud_type` is only used to filter training rows, never as a feature). We then check how many `card_testing` frauds of the test period it still flags.

In [9]:
keep_fit = fit.fraud_type.ne("card_testing").to_numpy()
keep_val = val.fraud_type.ne("card_testing").to_numpy()
params_nocard = {**XGB_PARAMS, "scale_pos_weight": float((y_fit[keep_fit] == 0).sum() / (y_fit[keep_fit] == 1).sum())}
xgb_nocard = xgb.XGBClassifier(**params_nocard)
xgb_nocard.fit(X_fit[keep_fit], y_fit[keep_fit], eval_set=[(X_val[keep_val], y_val[keep_val])], verbose=False)
thr_nocard = best_threshold(y_val[keep_val], xgb_nocard.predict_proba(X_val[keep_val])[:, 1])
pred_nocard = xgb_nocard.predict_proba(X_test)[:, 1] >= thr_nocard

card = (test.fraud_type == "card_testing").to_numpy()
pd.DataFrame({
    "card_testing recall (test)": [f"{pred_xgb[card].sum()}/{card.sum()} = {pred_xgb[card].mean():.0%}",
                                   f"{pred_nocard[card].sum()}/{card.sum()} = {pred_nocard[card].mean():.0%}"],
    "other frauds recall (test)": [f"{pred_xgb[y_test.to_numpy().astype(bool) & ~card].mean():.0%}",
                                   f"{pred_nocard[y_test.to_numpy().astype(bool) & ~card].mean():.0%}"],
    "F1 on all test": [f1_score(y_test, pred_xgb), f1_score(y_test, pred_nocard)],
}, index=["XGBoost (all scenarios)", "XGBoost (no card_testing in training)"])

,card_testing recall (test),other frauds recall (test),F1 on all test
XGBoost (all scenarios),91/94 = 97%,62%,0.856230
XGBoost (no card_testing in training),0/94 = 0%,78%,0.477876


## 8. Save the models

In [10]:
models_dir = ML_DIR / "models"
models_dir.mkdir(exist_ok=True)
model_xgb.save_model(models_dir / "xgb_fraud.json")
joblib.dump({"pipeline": logreg, "medians": MEDIANS, "log_cols": LOG_COLS}, models_dir / "logreg_fraud.joblib")
metadata = {
    "features": FEATURES, "numeric": NUMERIC, "categorical": CATEGORIES,
    "thresholds": {"xgboost": thr_xgb, "logistic_regression": thr_lr},
    "train_period": [str(train.transaction_time.min()), str(train.transaction_time.max())],
    "validation_from": str(cut),
    "test_metrics": {name: {k: float(v) for k, v in row.items()} for name, row in results.iterrows()},
    "xgb_params": dict(XGB_PARAMS.items()), "xgb_best_iteration": int(model_xgb.best_iteration),
}
(models_dir / "model_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
sorted(p.name for p in models_dir.iterdir())

['logreg_fraud.joblib', 'model_metadata.json', 'xgb_fraud.json']